# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
data = pd.read_csv("AviationData.csv", encoding ="latin1")
data.info()
data.head()

/var/folders/p8/wb_rm8f14572pdbjcnwxgr240000gn/T/ipykernel_24468/322886731.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("AviationData.csv", encoding ="latin1")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# inspect the NaN values in the dataset
print(data.isna().sum())

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

In [4]:
print(data.describe())

       Number.of.Engines  Total.Fatal.Injuries  Total.Serious.Injuries  \
count       82805.000000          77488.000000            76379.000000   
mean            1.146585              0.647855                0.279881   
std             0.446510              5.485960                1.544084   
min             0.000000              0.000000                0.000000   
25%             1.000000              0.000000                0.000000   
50%             1.000000              0.000000                0.000000   
75%             1.000000              0.000000                0.000000   
max             8.000000            349.000000              161.000000   

       Total.Minor.Injuries  Total.Uninjured  
count          76956.000000     82977.000000  
mean               0.357061         5.325440  
std                2.235625        27.913634  
min                0.000000         0.000000  
25%                0.000000         0.000000  
50%                0.000000         1.000000  
75% 

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [5]:
# inspect relevant columns
relevant_cols = ["Event.Date", "Aircraft.Category", "Make", "Model", "Amateur.Built"]
print(data[relevant_cols].info())
print(data[relevant_cols].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Event.Date         88889 non-null  object
 1   Aircraft.Category  32287 non-null  object
 2   Make               88826 non-null  object
 3   Model              88797 non-null  object
 4   Amateur.Built      88787 non-null  object
dtypes: object(5)
memory usage: 3.4+ MB
None
Event.Date               0
Aircraft.Category    56602
Make                    63
Model                   92
Amateur.Built          102
dtype: int64


In [6]:
# check unique values in the columns
print(data["Aircraft.Category"].unique())
print(data["Amateur.Built"].unique())

[nan 'Airplane' 'Helicopter' 'Glider' 'Balloon' 'Gyrocraft' 'Ultralight'
 'Unknown' 'Blimp' 'Powered-Lift' 'Weight-Shift' 'Powered Parachute'
 'Rocket' 'WSFT' 'UNK' 'ULTR']
['No' 'Yes' nan]


In [7]:
# Keep only the airplanes
data = data[data["Aircraft.Category"] == "Airplane"]

# The missing amateur built values are assumed to be no
data["Amateur.Built"] = data["Amateur.Built"].fillna("No")

# We need professionally built aircraft only
data = data[data["Amateur.Built"] == "No"]

# drop the columns in which we cannot identify the make or model of the jets
data = data.dropna(subset=["Make", "Model"])

In [8]:
# Filter the data
# filter by years to satisfy assumption of max jet lifetime, i.e 40 years

data["Event.Date"].isna().sum()

# convert to datetime as it is still an object
data["Event.Date"] = pd.to_datetime(data["Event.Date"])

# Then filter the column
data = data.dropna(subset=["Event.Date"])
data = data[data["Event.Date"].dt.year >= 1983]

In [9]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
Index: 21444 entries, 4149 to 88886
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                21444 non-null  object        
 1   Investigation.Type      21444 non-null  object        
 2   Accident.Number         21444 non-null  object        
 3   Event.Date              21444 non-null  datetime64[ns]
 4   Location                21437 non-null  object        
 5   Country                 21443 non-null  object        
 6   Latitude                19164 non-null  object        
 7   Longitude               19158 non-null  object        
 8   Airport.Code            13976 non-null  object        
 9   Airport.Name            14061 non-null  object        
 10  Injury.Severity         20633 non-null  object        
 11  Aircraft.damage         20216 non-null  object        
 12  Aircraft.Category       21444 non-null  object  

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

In [10]:
# Selecting the injury and damage columns, create a list
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]

# Clean the injury columns
# first, convert data type to numeric, handle errors using coerce because we asssume non numeric text is missing data
for column in injury_cols:
    data[column] = pd.to_numeric(data[column], errors="coerce")

# impute missing values as zero
data[injury_cols] = data[injury_cols].fillna(0)

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [11]:
# Estimating the total passengers: we assume that injury categories provide an accurate 
# estimate of the number of passengers. Create a new column in the original data
data["Total.Passengers"] = data[injury_cols].sum(axis = 1)

# Retain the rows that have values greater than 0
data = data[data["Total.Passengers"] > 0]
data.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,0.0,0.0,0.0,588.0,VMC,Standing,Probable Cause,04-12-2014,588.0
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,"Singapore Airlines, Ltd.",0.0,0.0,0.0,588.0,VMC,Taxi,Probable Cause,04-12-2014,588.0
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,NaN,1.0,1.0,0.0,0.0,IMC,Cruise,Probable Cause,02-05-2011,2.0
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,NaN,1.0,0.0,0.0,4.0,VMC,Standing,Probable Cause,17-10-2016,5.0
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,289.0,VMC,Cruise,Probable Cause,01-02-2016,289.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
# Inspect the values in the category "Airplane.damage"
print(data["Aircraft.damage"].value_counts(dropna=False))

# First, replace the unknown with NaN, and then drop the columns.
# This is because of the assumption that unknown or missing values cannot be relied upon without introducing a bias
data["Aircraft.damage"] = data["Aircraft.damage"].replace("Unknown", pd.NA)
data = data.dropna(subset=["Aircraft.damage"])

Aircraft.damage
Substantial    16816
Destroyed       2266
NaN              786
Minor            595
Unknown           74
Name: count, dtype: int64


In [13]:
# looking at the fatality rate
data["Fatality.Rate"] = (data["Total.Fatal.Injuries"]/data["Total.Passengers"])

# Rate of serious injuries
data["Serious.Injury.Rate"] = (data["Total.Serious.Injuries"]/data["Total.Passengers"])

# Rate of severe risk of injury
data["Severe.Risk"] = ((data["Total.Fatal.Injuries"] + data["Total.Serious.Injuries"])/ data["Total.Passengers"])

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [14]:
# inspecting the make column 
print(data["Make"].head())
print(data["Make"].nunique())
print(data["Make"].isna().sum())
print(data["Make"].value_counts().head(8))

# Cleaning tasks
# Standardize the text
data["Make"] = data["Make"].str.strip()  # this will remove spaces
data["Make"] = data["Make"].str.title()  # this will standardize case to capitalize the first letter of every word
print(data["Make"].value_counts().head(5))

# Apply the threshold of 50 records
count_make = data["Make"].value_counts()
valid_makes = count_make[count_make >= 50].index
data = data[data["Make"].isin(valid_makes)]
print(data["Make"].value_counts().head(5))

4149    Lockheed
4150      Boeing
4171       Piper
5957     Douglas
5960    Lockheed
Name: Make, dtype: object
1289
0
Make
CESSNA    4691
PIPER     2739
Cessna    2253
Piper     1180
BEECH      987
Beech      405
BOEING     363
MOONEY     230
Name: count, dtype: int64
Make
Cessna    6944
Piper     3919
Beech     1392
Boeing     482
Mooney     354
Name: count, dtype: int64
Make
Cessna    6944
Piper     3919
Beech     1392
Boeing     482
Mooney     354
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [15]:
# inspect the Model column
print(data["Model"].head())
print(data["Model"].nunique())
print(data["Model"].isna().sum())
print(data["Model"].value_counts().head(8))

# check if models are unique to each make
model_make_count = data.groupby("Model")["Make"].nunique()
model_make_count[model_make_count > 1].head() # observe that the same model appear multiple times under different makes

# since models are not unique to plane types, creating a unique identifier
data = data.copy(deep=True)
data["Aircraft.Type"] = data[["Make", "Model"]].agg(" ".join, axis = 1)
data["Aircraft.Type"].value_counts()

4150          747
4171    PA-28-140
6760      727-200
6806          C35
7084         180K
Name: Model, dtype: object
1806
0
Model
172     742
152     306
182     298
172S    266
PA28    262
172N    244
SR22    231
180     212
Name: count, dtype: int64


Aircraft.Type
Cessna 172             742
Cessna 152             306
Cessna 182             298
Cessna 172S            266
Piper PA28             262
                      ... 
Boeing MD                1
Cessna 550B              1
North American T6        1
North American P51D      1
Piper PA-44              1
Name: count, Length: 1905, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [16]:
# inspecting the columns in the data
columns = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]
for col in columns:
    print(f"\n {column} ")
    print(data[col].value_counts(dropna=False).head(10))


 Total.Uninjured 
Engine.Type
Reciprocating      12747
NaN                 2249
Turbo Prop           882
Turbo Fan            323
Unknown               65
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

 Total.Uninjured 
Weather.Condition
VMC    13919
NaN     1383
IMC      846
Unk      128
UNK       41
Name: count, dtype: int64

 Total.Uninjured 
Number.of.Engines
1.0    13083
2.0     1887
NaN     1292
4.0       36
3.0       16
0.0        3
Name: count, dtype: int64

 Total.Uninjured 
Purpose.of.flight
Personal              9773
Instructional         2378
NaN                   1696
Aerial Application     721
Business               400
Positioning            255
Unknown                244
Skydiving              153
Aerial Observation     145
Other Work Use         117
Name: count, dtype: int64

 Total.Uninjured 
Broad.phase.of.flight
NaN            13896
Landing         1109
Takeoff          423
Cruise      

In [17]:
# 1. Engine Type
# Replace the placeholder values, such as UNK, Unk and Unknown
data["Engine.Type"] = data["Engine.Type"].str.strip().str.title()
data["Engine.Type"] = data["Engine.Type"].replace(["Unknown", "Unk", "UNK"], pd.NA)

# Check the types of engines with too few examples to use, in this case lessa than 50
engine_count = data["Engine.Type"].value_counts()
few_engines = engine_count[engine_count < 50].index
# put them in a new category, other
data.loc[data["Engine.Type"].isin(few_engines), "Engine.Type"] = "Other"


In [35]:
# 2. Weather Condition
# Replace the placeholder values, such as UNK, Unk and Unknown
data["Weather.Condition"] = data["Weather.Condition"].str.strip()
data["Weather.Condition"] = data["Weather.Condition"].replace(["Unknown", "Unk", "UNK"], pd.NA)
data["Weather.Condition"] = data["Weather.Condition"].fillna(pd.NA)

In [36]:
# 3. Number of Engines
# Change the data type to numeric
data["Number.of.Engines"] = pd.to_numeric(data["Number.of.Engines"], errors="coerce")
data["Engine.Type"] = data["Engine.Type"].fillna(pd.NA)
# Remove the invalid values
data= data[data["Number.of.Engines"] > 0]

In [37]:
# 4. Purpose of flight
# Replace the placeholder values, such as UNK, Unk and Unknown
data["Purpose.of.flight"] = data["Purpose.of.flight"].str.strip()
data["Purpose.of.flight"] = data["Purpose.of.flight"].replace(["Unknown", "UNK"], pd.NA)
data["Purpose.of.flight"] = data["Purpose.of.flight"].fillna(pd.NA)
# Check the purpose/reason for flight
purpose_count = data["Purpose.of.flight"].value_counts()
unique_purpose = purpose_count[purpose_count < 100].index
# group into a new category, other
data.loc[data["Purpose.of.flight"].isin(unique_purpose), "Purpose.of.flight"] = "Other"


In [38]:
# Broad phase of flight
data["Broad.phase.of.flight"] = data["Broad.phase.of.flight"].str.strip()
data["Broad.phase.of.flight"] = data["Broad.phase.of.flight"].fillna("Unknown")

In [39]:
columns = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]
for col in columns:
    print(f"\n {column} ")
    print(data[col].value_counts(dropna=False).head(10))


 Total.Uninjured 
Engine.Type
Reciprocating    12533
<NA>              1303
Turbo Prop         842
Turbo Fan          297
Other               47
Name: count, dtype: int64

 Total.Uninjured 
Weather.Condition
VMC     13486
IMC       785
<NA>      751
Name: count, dtype: int64

 Total.Uninjured 
Number.of.Engines
1.0    13083
2.0     1887
4.0       36
3.0       16
Name: count, dtype: int64

 Total.Uninjured 
Purpose.of.flight
Personal              9470
Instructional         2323
<NA>                  1125
Aerial Application     660
Other                  421
Business               380
Positioning            245
Skydiving              150
Aerial Observation     138
Other Work Use         110
Name: count, dtype: int64

 Total.Uninjured 
Broad.phase.of.flight
Unknown        12617
Landing         1106
Takeoff          422
Cruise           232
Approach         208
Maneuvering      127
Taxi              94
Go-around         81
Descent           57
Climb             50
Name: count, dtype: int6

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# inspect the columns
data.info()
 # the columns in my data frame have less than 20,000 observations, so I did not filter anymore

<class 'pandas.core.frame.DataFrame'>
Index: 15022 entries, 4150 to 88886
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                15022 non-null  object        
 1   Investigation.Type      15022 non-null  object        
 2   Accident.Number         15022 non-null  object        
 3   Event.Date              15022 non-null  datetime64[ns]
 4   Location                15022 non-null  object        
 5   Country                 15021 non-null  object        
 6   Latitude                14598 non-null  object        
 7   Longitude               14594 non-null  object        
 8   Airport.Code            10761 non-null  object        
 9   Airport.Name            10863 non-null  object        
 10  Injury.Severity         15022 non-null  object        
 11  Aircraft.damage         15022 non-null  object        
 12  Aircraft.Category       15022 non-null  object  

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [42]:
data.to_csv("cleaned_aviation_data.csv", index=False)